# ira-esm-tokenizer on Kaggle

Self-contained. Every source file is written out by the cells below, so
nothing needs cloning or uploading.

**Before you run anything**, set these in the right-hand sidebar:

- **Accelerator** -> GPU T4 x2
- **Internet** -> On  (needed for pip and for downloading PDB files)

Changing either of those restarts the kernel and wipes all state, so set
them first, then Run All.

## 1. Check the session

In [ ]:
import torch, subprocess

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
else:
    print("NO GPU. Set Accelerator to GPU T4 x2 in the sidebar (this restarts the kernel).")

# Internet check -- if this fails, flip Internet on in the sidebar.
import urllib.request
try:
    urllib.request.urlopen("https://files.rcsb.org", timeout=10)
    print("internet: ok")
except Exception as e:
    print("internet: OFF ->", e)

## 2. Install dependencies

In [ ]:
!pip install -q biopython
print('done')

## 3. Write out the source files

The project lives in `/kaggle/working/ira-esm-tokenizer` so that checkpoints
land in Kaggle's output and survive the session.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path("/kaggle/working/ira-esm-tokenizer")
for sub in ["model", "data", "checkpoints"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("working in", os.getcwd())

In [ ]:
%%writefile model/geometry.py
"""
Rigid-body reference frames per residue, built from backbone coordinates.

This is the geometric foundation the encoder is built on (same idea as
AlphaFold's "rigidFrom3Points" and ESM3's structure encoder): each residue
gets its own local coordinate frame (a rotation + a translation) derived
from its own N, CA, C atom positions. Expressing other residues relative
to this frame, instead of in absolute world coordinates, is what makes the
resulting features invariant to how the whole protein is rotated or moved
in space -- the same fold produces the same features regardless of its
orientation.
"""

import torch

# Indices into the 4-atom backbone dimension (N, CA, C, O) used throughout
# the data pipeline -- see BACKBONE_ATOMS in parse_structures.py.
N_IDX, CA_IDX, C_IDX, O_IDX = 0, 1, 2, 3


def build_frames(coords: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Build one rigid frame per residue from its N, CA, C atoms.

    coords: (B, L, 4, 3) -- batch of structures, backbone atoms per residue.
    Returns (rotations, translations):
      rotations:    (B, L, 3, 3) rotation matrix per residue
      translations: (B, L, 3)    origin (= that residue's CA position)

    The frame is centered at CA, with its axes built from the N-CA and
    C-CA directions via Gram-Schmidt -- this is exactly how AlphaFold
    defines each residue's local frame from three backbone points.
    """
    n = coords[..., N_IDX, :]   # (B, L, 3)
    ca = coords[..., CA_IDX, :]  # (B, L, 3)
    c = coords[..., C_IDX, :]   # (B, L, 3)

    # The frame's origin is the residue's own CA position -- every other
    # residue's position will later be described as "how far from this
    # residue's CA, in this residue's own rotated axes."
    translations = ca

    # First axis: the direction from CA toward N, normalized to unit length.
    # The .clamp(min=1e-8) on every norm below is not cosmetic. collate_fn
    # pads short structures with ZERO coordinates, so a padded position has
    # n == ca == c == 0, giving a zero-length v1 and a 0/0 = NaN axis. NaN
    # then spreads everywhere downstream, including into real residues (it
    # survives being multiplied by a zero attention weight), so masking
    # alone does not contain it. Clamping makes padded frames harmlessly
    # zero instead of NaN, and the masks discard them as intended.
    v1 = n - ca
    e1 = v1 / v1.norm(dim=-1, keepdim=True).clamp(min=1e-8)

    # Second axis: start from the CA->C direction, then Gram-Schmidt it --
    # subtract out whatever part of it points along e1, so e1 and e2 end
    # up exactly perpendicular. Without this step the two axes wouldn't
    # form a valid (orthogonal) coordinate system.
    v2 = c - ca
    v2 = v2 - (v2 * e1).sum(dim=-1, keepdim=True) * e1
    e2 = v2 / v2.norm(dim=-1, keepdim=True).clamp(min=1e-8)

    # Third axis: perpendicular to both e1 and e2, via the cross product --
    # completes a valid right-handed 3D coordinate system.
    e3 = torch.cross(e1, e2, dim=-1)

    # Stack the three axes as columns to form each residue's rotation matrix.
    rotations = torch.stack([e1, e2, e3], dim=-1)  # (B, L, 3, 3)

    return rotations, translations


def to_local_frame(
    point: torch.Tensor, rotations: torch.Tensor, translations: torch.Tensor
) -> torch.Tensor:
    """
    Express world-coordinate point(s) relative to each residue's own frame.

    point:        (B, L, 3) or (B, L, L, 3) -- world-space position(s)
    rotations:    (B, L, 3, 3) from build_frames
    translations: (B, L, 3)    from build_frames

    This answers "where does this point sit, if I measure it using this
    residue's own rotated axes, starting from this residue's own CA?" --
    the actual invariant feature. Rotate or translate the whole protein
    in world space, and this output stays identical.
    """
    # Handle both a single point per residue (B, L, 3) and a full pairwise
    # grid (B, L, L, 3, one entry per residue-pair) by inserting a
    # broadcastable dimension for the pairwise case.
    if point.dim() == translations.dim():
        # (B, L, 3): shift into this frame's origin, then unsqueeze for matmul
        centered = (point - translations).unsqueeze(-1)  # (B, L, 3, 1)
        local = rotations.transpose(-1, -2) @ centered     # (B, L, 3, 1)
        return local.squeeze(-1)                            # (B, L, 3)
    else:
        # (B, L, L, 3): pairwise -- translations/rotations broadcast over
        # the extra L dimension (each row uses its own residue's frame)
        centered = (point - translations.unsqueeze(2)).unsqueeze(-1)  # (B, L, L, 3, 1)
        rot = rotations.unsqueeze(2).transpose(-1, -2)                 # (B, L, 1, 3, 3)
        local = rot @ centered                                          # (B, L, L, 3, 1)
        return local.squeeze(-1)                                        # (B, L, L, 3)


In [ ]:
%%writefile model/encoder.py
"""
Structure encoder: turns per-residue backbone geometry into one continuous
feature vector per residue, ready to be quantized into discrete tokens.

Deliberately does NOT see the amino acid sequence -- only geometry -- so
that the same fold produces similar features regardless of which amino
acids happen to compose it. This is the actual "understanding shape"
part of the pipeline; VQ quantization (next file) turns these continuous
vectors into discrete tokens.
"""

import torch
import torch.nn as nn

from model.geometry import build_frames, to_local_frame


def pairwise_features(coords: torch.Tensor) -> torch.Tensor:
    """
    Build a feature vector for every (residue_i, residue_j) pair, describing
    residue j's geometry as seen from residue i's own local frame.

    coords: (B, L, 4, 3)
    Returns: (B, L, L, 13) -- for each pair, 3 numbers for relative position,
    9 for relative orientation, 1 for distance.
    """
    rotations, translations = build_frames(coords)  # (B,L,3,3), (B,L,3)
    B, L, _ = translations.shape

    # Relative position: where does residue j's CA sit, measured in
    # residue i's own local axes?
    ca = coords[:, :, 1, :]                       # (B, L, 3) -- CA atoms only
    ca_j = ca.unsqueeze(1).expand(-1, L, -1, -1)   # (B, L, L, 3): j varies along dim 2
    relative_position = to_local_frame(ca_j, rotations, translations)  # (B, L, L, 3)

    # Relative orientation: how is residue j's own frame rotated, compared
    # to residue i's frame? R_i^T @ R_j -- invariant for the same reason
    # relative_position is (a shared whole-protein rotation cancels out).
    rot_i = rotations.unsqueeze(2)                 # (B, L, 1, 3, 3)
    rot_j = rotations.unsqueeze(1)                 # (B, 1, L, 3, 3)
    relative_rotation = rot_i.transpose(-1, -2) @ rot_j  # (B, L, L, 3, 3)
    relative_rotation_flat = relative_rotation.reshape(B, L, L, 9)

    # Distance between residues -- redundant with relative_position (it's
    # that vector's length) but giving it directly speeds up training.
    distance = relative_position.norm(dim=-1, keepdim=True)  # (B, L, L, 1)

    return torch.cat([relative_position, relative_rotation_flat, distance], dim=-1)
    # final shape: (B, L, L, 13)


class GeometricAttentionLayer(nn.Module):
    """
    One layer of attention where the pairwise geometry between residues
    does two separate jobs:
      1. biases WHICH residues a given residue pays attention to
         (pair_to_bias), and
      2. contributes its own content to WHAT gets aggregated
         (pair_to_value).

    Job 2 matters more than it looks: every residue starts from the exact
    same learned vector (see StructureEncoder.initial_embedding below), so
    without job 2, every residue would be averaging together identical
    values -- and averaging identical values always gives back that same
    identical value, no matter how the averaging is weighted. Residues
    would never become distinguishable from one another. pair_to_value
    injects real, pair-specific geometric content directly into the
    aggregation, which is what actually breaks that symmetry.
    """

    def __init__(self, dim: int, num_heads: int, pair_dim: int = 13):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, dim)
        self.to_v = nn.Linear(dim, dim)
        self.to_out = nn.Linear(dim, dim)

        # Turns each pair's 13-number geometry into one attention-logit
        # bias per head -- how much should this pair's real 3D relationship
        # push residue i to attend to residue j.
        self.pair_to_bias = nn.Linear(pair_dim, num_heads)

        # Turns each pair's 13-number geometry into a full per-head content
        # vector -- the "sticker" added onto residue j's value before it's
        # folded into residue i's blend. This is what makes the blended
        # result differ across residues even when every residue's own
        # value vector started out identical.
        self.pair_to_value = nn.Linear(pair_dim, dim)

        self.norm = nn.LayerNorm(dim)

        # Xavier (Glorot) initialization: picks each layer's starting random
        # weights based on both its input and output size, so signal
        # variance stays roughly stable as data passes through many stacked
        # layers. PyTorch's nn.Linear default is Kaiming/He initialization,
        # which is tuned more for ReLU-style networks -- Xavier is the more
        # standard choice for layers without a ReLU in between, which is
        # what we have here (pure attention, no feed-forward block yet).
        # Biases start at zero, standard practice -- Xavier's variance
        # formula is specifically about the weight matrix, not the bias.
        for layer in [self.to_q, self.to_k, self.to_v, self.to_out, self.pair_to_bias, self.pair_to_value]:
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor, pair_feats: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        x:          (B, L, dim)   -- current per-residue features
        pair_feats: (B, L, L, 13) -- from pairwise_features()
        mask:       (B, L)        -- True for real residues, False for padding
        """
        B, L, dim = x.shape
        x_norm = self.norm(x)

        q = self.to_q(x_norm).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)  # (B, H, L, hd)
        k = self.to_k(x_norm).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.to_v(x_norm).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        attn_logits = q @ k.transpose(-1, -2) / (self.head_dim ** 0.5)  # (B, H, L, L)

        pair_bias = self.pair_to_bias(pair_feats).permute(0, 3, 1, 2)  # (B, H, L, L)
        attn_logits = attn_logits + pair_bias

        pad_mask = mask.unsqueeze(1).unsqueeze(1)  # (B, 1, 1, L)
        attn_logits = attn_logits.masked_fill(~pad_mask, float("-inf"))
        attn_weights = attn_logits.softmax(dim=-1)  # (B, H, L, L)

        # Per-pair "sticker" content: reshape into per-head chunks so it
        # can be added onto v, which is also split per head.
        pair_values = self.pair_to_value(pair_feats)  # (B, L, L, dim)
        pair_values = pair_values.view(B, L, L, self.num_heads, self.head_dim).permute(0, 3, 1, 2, 4)
        # pair_values: (B, H, L, L, hd) -- one sticker-adjusted value per
        # (query residue i, key residue j) pair, per head.

        # v is the same for every query residue i (it only varies by key
        # residue j), so broadcast it across the new "i" dimension before
        # adding the pair-specific stickers, which DO vary by i.
        v_expanded = v.unsqueeze(2).expand(-1, -1, L, -1, -1)  # (B, H, L, L, hd)
        combined_values = v_expanded + pair_values               # (B, H, L, L, hd)

        # Weighted blend: for each query residue i, combine every key
        # residue j's (value + sticker) using the attention weights.
        attn_out = (attn_weights.unsqueeze(-1) * combined_values).sum(dim=3)  # (B, H, L, hd)

        attn_out = attn_out.transpose(1, 2).reshape(B, L, dim)
        out = self.to_out(attn_out)

        # Residual connection: refine the input rather than replace it,
        # standard practice for stable training across stacked layers.
        return x + out


class StructureEncoder(nn.Module):
    """
    Stacks several GeometricAttentionLayers to turn raw backbone geometry
    into a final per-residue feature vector, ready for VQ quantization.
    """

    def __init__(self, dim: int = 128, num_heads: int = 4, num_layers: int = 4):
        super().__init__()

        # Every residue starts from the same learned vector -- there's no
        # sequence information to differentiate them at the input. All
        # differentiation between residues comes from geometry, injected
        # via pair_to_value inside each attention layer.
        self.initial_embedding = nn.Parameter(torch.randn(dim) * 0.02)

        self.layers = nn.ModuleList(
            [GeometricAttentionLayer(dim, num_heads) for _ in range(num_layers)]
        )

    def forward(self, coords: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        coords: (B, L, 4, 3)
        mask:   (B, L)
        Returns: (B, L, dim) -- one feature vector per residue
        """
        B, L = mask.shape
        pair_feats = pairwise_features(coords)  # (B, L, L, 13), computed once, reused every layer

        x = self.initial_embedding.expand(B, L, -1)
        for layer in self.layers:
            x = layer(x, pair_feats, mask)

        return x


In [ ]:
%%writefile model/quantizer.py
"""
Vector quantizer: the actual "tokenizer" step. Snaps each residue's
continuous 128-number encoder output to the nearest entry in a fixed,
learned codebook, producing a discrete integer per residue -- the
structure token.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class VectorQuantizer(nn.Module):
    """
    num_codes: size of the codebook -- how many distinct structure tokens
    can exist. 4096 matches ESM3's actual structure tokenizer; big enough
    to capture a wide variety of local shapes, small enough to keep the
    vocabulary learnable from ~1000 training structures.

    code_dim: must match the encoder's output size (dim=128), since we're
    comparing encoder outputs directly against codebook entries.
    """

    def __init__(self, num_codes: int = 4096, code_dim: int = 128, commitment_weight: float = 0.25):
        super().__init__()
        self.commitment_weight = commitment_weight

        # The codebook itself: num_codes learned vectors, each code_dim
        # numbers long -- literally the "4096 reference paint chips."
        # These start as random noise and get shaped by training into
        # genuinely useful, distinct local-geometry prototypes.
        self.codebook = nn.Embedding(num_codes, code_dim)

        # Small random initialization, same reasoning as the encoder's
        # initial_embedding -- keeps early training numerically stable.
        nn.init.normal_(self.codebook.weight, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> dict:
        """
        x:    (B, L, code_dim) -- continuous encoder output
        mask: (B, L)           -- True for real residues, False for padding

        Returns a dict with:
          tokens:     (B, L) integer token id per residue -- the actual output
          quantized:  (B, L, code_dim) the snapped-to-codebook vectors,
                      usable as input to the decoder
          loss:       scalar, added to the training loss (see below)
        """
        B, L, D = x.shape
        codebook = self.codebook.weight  # (num_codes, code_dim)

        # Find each residue's nearest codebook entry, measured by squared
        # Euclidean distance -- "which reference chip is the closest color
        # match." Expand x and codebook to compare every residue against
        # every code in one batched operation, rather than looping.
        x_flat = x.reshape(B * L, D)                                # (B*L, D)
        distances = torch.cdist(x_flat, codebook)                    # (B*L, num_codes)
        tokens_flat = distances.argmin(dim=1)                        # (B*L,) -- nearest code index per residue
        tokens = tokens_flat.view(B, L)

        # Look up the actual vector for each chosen code -- this is what
        # gets passed on to the decoder, not the raw continuous x.
        quantized = self.codebook(tokens)  # (B, L, D)

        # --- The straight-through estimator ---
        # quantized.detach() means "treat this as a fixed constant during
        # backpropagation, don't try to compute its gradient." Adding
        # (x - x.detach()), which numerically equals zero, doesn't change
        # the actual value at all -- but it does change what gradients
        # flow backward: gradients end up flowing as if this whole line
        # were just "quantized = x", letting training signal reach the
        # encoder even though argmin itself has no usable gradient.
        quantized_st = x + (quantized - x).detach()

        # --- Two loss terms, both standard VQ-VAE ingredients ---
        # Both computed per-residue, then averaged only over REAL
        # residues, ignoring padding -- same masking principle as
        # everywhere else in this pipeline.
        mask_flat = mask.reshape(-1).float()

        # Codebook loss: pushes each chosen codebook vector to move
        # closer to the encoder outputs that picked it -- literally
        # "adjust the reference chip's color to better match the custom
        # colors that keep getting matched to it."
        per_residue_codebook_loss = F.mse_loss(
            quantized.reshape(-1, D), x.detach().reshape(-1, D), reduction="none"
        ).mean(dim=-1)
        codebook_loss = (per_residue_codebook_loss * mask_flat).sum() / mask_flat.sum()

        # Commitment loss: the reverse pull -- pushes the encoder's
        # output to move closer to whichever codebook vector it already
        # picked, discouraging the encoder from producing wildly varying
        # outputs that hop between different codes for similar inputs.
        # Weighted down (0.25) because this term is a stabilizer, not the
        # main learning signal -- weighting it too heavily can make the
        # encoder collapse to outputting near-identical values everywhere.
        per_residue_commitment_loss = F.mse_loss(
            x.reshape(-1, D), quantized.detach().reshape(-1, D), reduction="none"
        ).mean(dim=-1)
        commitment_loss = (per_residue_commitment_loss * mask_flat).sum() / mask_flat.sum()

        loss = codebook_loss + self.commitment_weight * commitment_loss

        return {
            "tokens": tokens,
            "quantized": quantized_st,
            "loss": loss,
        }


In [ ]:
%%writefile model/decoder.py
"""
Structure decoder: turns the discrete structure tokens back into 3D backbone
coordinates. This is the half that makes the tokens mean anything.

Without a decoder the codebook is unconstrained -- the encoder and quantizer
could happily settle on 4096 codes that carve up their own feature space
neatly but have nothing to do with actual shape. Forcing the tokens to be
sufficient to REBUILD the backbone is what makes them encode geometry.

Two design consequences follow from how the encoder was built, and they
drive almost every decision in this file:

1. The encoder never sees the amino acid sequence, and it never sees
   absolute positions -- only rotation-invariant pairwise geometry. So the
   tokens genuinely do not contain the protein's orientation in space.
   The decoder therefore CANNOT be trained against raw coordinates; it has
   no way to know which way the protein was pointing. The loss has to be
   orientation-blind too, which is why this file ends in FAPE rather than
   a plain mean-squared error on xyz.

2. The encoder got all of its residue-to-residue structure from geometric
   pair features. The decoder has no geometry to start from (that's the
   thing it's trying to produce), so it uses an ordinary transformer over
   the token sequence instead, plus positional encoding to know chain order.
"""

import math

import torch
import torch.nn as nn

from model.geometry import build_frames, to_local_frame

# Ideal backbone geometry, in Angstroms and degrees. These are the
# textbook average values for a protein backbone, and they barely vary
# between residues in real structures -- bond lengths and bond angles are
# essentially fixed by chemistry. Only the TORSIONS (how the chain rotates
# about its bonds) actually differ from fold to fold, so those are the
# only part worth spending network capacity on predicting.
BOND_N_CA = 1.458
BOND_CA_C = 1.525
BOND_C_O = 1.231
ANGLE_N_CA_C = math.radians(111.2)
ANGLE_CA_C_O = math.radians(120.8)


def ideal_local_backbone() -> dict:
    """
    Where the N, CA, C atoms of a single residue sit inside that residue's
    OWN local frame, assuming ideal chemistry.

    This is fully determined, with no freedom left, because build_frames()
    defines the local frame from those exact three atoms:
      - the origin is CA, so CA is at (0, 0, 0)
      - the first axis points at N, so N lies on the positive x-axis
      - the second axis is the Gram-Schmidt leftover of CA->C, so C lies
        in the xy-plane with no z-component at all

    In other words, once you know a residue's frame you already know where
    its N, CA and C are. The network only has to predict the frame.

    Also returns an orthonormal basis (u, p, q) at the C atom, used for
    placing O -- see place_oxygen() for why O is the odd one out.
    """
    ca = torch.zeros(3)
    n = torch.tensor([BOND_N_CA, 0.0, 0.0])
    c = torch.tensor([
        BOND_CA_C * math.cos(ANGLE_N_CA_C),
        BOND_CA_C * math.sin(ANGLE_N_CA_C),
        0.0,
    ])

    # u: unit vector along the CA->C bond. O sits at a fixed angle away
    # from this axis, and is free to swing around it.
    u = c / c.norm()

    # p, q: two unit vectors perpendicular to u and to each other, giving
    # a "clock face" around the CA->C axis. An angle on this clock face is
    # the one genuinely free number in O's position.
    n_dir = n / n.norm()
    p = n_dir - (n_dir @ u) * u   # strip out the part pointing along u
    p = p / p.norm()
    q = torch.cross(u, p, dim=0)

    return {"n": n, "ca": ca, "c": c, "u": u, "p": p, "q": q}


def rotation_from_6d(v: torch.Tensor) -> torch.Tensor:
    """
    Turn 6 freely-predicted numbers into a valid rotation matrix.

    v: (..., 6)  ->  (..., 3, 3)

    A network can't just output 9 numbers and call it a rotation, because
    almost no 9 numbers form one (they have to be orthonormal with
    determinant +1). Angles are no better: Euler angles and quaternions
    both have discontinuities, points where a tiny change in the true
    rotation demands a huge jump in the predicted numbers, which networks
    learn very badly.

    The fix (Zhou et al. 2019) is to predict two arbitrary 3D vectors and
    Gram-Schmidt them into a frame -- exactly the same construction
    build_frames() uses on N/CA/C. Every possible input maps to a valid
    rotation, and nearby rotations always have nearby inputs.
    """
    a, b = v[..., :3], v[..., 3:]

    e1 = a / a.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    b = b - (b * e1).sum(dim=-1, keepdim=True) * e1
    e2 = b / b.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    e3 = torch.cross(e1, e2, dim=-1)

    # Stacked as COLUMNS, matching build_frames' convention, so the two
    # kinds of frame are interchangeable everywhere downstream.
    return torch.stack([e1, e2, e3], dim=-1)


def sinusoidal_positions(length: int, dim: int, device, dtype) -> torch.Tensor:
    """
    Standard sine/cosine positional encoding, (length, dim).

    The decoder needs this because a plain transformer is order-blind: shuffle
    its inputs and it produces the same outputs shuffled the same way. The
    encoder never needed positional encoding, since residue order was already
    implicit in the 3D geometry it was reading. Here there is no geometry yet,
    so chain order has to be supplied explicitly.

    Sinusoidal rather than learned because proteins vary in length and a
    learned table would cap how long a chain we can decode.
    """
    position = torch.arange(length, device=device, dtype=dtype).unsqueeze(1)
    freq = torch.exp(
        torch.arange(0, dim, 2, device=device, dtype=dtype) * (-math.log(10000.0) / dim)
    )
    pe = torch.zeros(length, dim, device=device, dtype=dtype)
    pe[:, 0::2] = torch.sin(position * freq)
    pe[:, 1::2] = torch.cos(position * freq)
    return pe


class TransformerLayer(nn.Module):
    """
    One ordinary pre-norm transformer block: self-attention, then a small
    feed-forward network, each wrapped in a residual connection.

    Note the difference from GeometricAttentionLayer in the encoder. That
    one had no feed-forward block, because its pair_to_value path was
    already injecting rich per-pair geometric content and doing the heavy
    lifting. Here there is no such path, so the feed-forward block is
    carrying the per-residue computation instead. Dropping it would leave
    the decoder able only to average token vectors together, which is far
    too weak to reconstruct a fold.
    """

    def __init__(self, dim: int, num_heads: int, ff_mult: int = 4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.norm_attn = nn.LayerNorm(dim)
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, dim)
        self.to_v = nn.Linear(dim, dim)
        self.to_out = nn.Linear(dim, dim)

        self.norm_ff = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim * ff_mult),
            nn.GELU(),
            nn.Linear(dim * ff_mult, dim),
        )

        # Xavier throughout, same reasoning as the encoder's layers.
        for layer in [self.to_q, self.to_k, self.to_v, self.to_out, self.ff[0], self.ff[2]]:
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        x:    (B, L, dim)
        mask: (B, L) -- True for real residues, False for padding
        """
        B, L, dim = x.shape

        h = self.norm_attn(x)
        q = self.to_q(h).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.to_k(h).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.to_v(h).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        attn_logits = q @ k.transpose(-1, -2) / (self.head_dim ** 0.5)  # (B, H, L, L)

        # Padding residues must never be attended TO, or real residues would
        # blend in meaningless zero-padding.
        attn_logits = attn_logits.masked_fill(~mask.unsqueeze(1).unsqueeze(1), float("-inf"))
        attn_weights = attn_logits.softmax(dim=-1)

        attn_out = (attn_weights @ v).transpose(1, 2).reshape(B, L, dim)
        x = x + self.to_out(attn_out)

        x = x + self.ff(self.norm_ff(x))
        return x


class StructureDecoder(nn.Module):
    """
    Tokens (as their quantized vectors) in, backbone coordinates out.

    The output head does NOT predict 12 loose xyz numbers per residue. It
    predicts each residue's rigid FRAME (a rotation and a position), then
    drops the atoms into that frame at their ideal chemical offsets. That
    way bond lengths and bond angles are correct by construction and can
    never drift, and the network spends its capacity on the part that
    actually varies between folds -- how each residue is oriented relative
    to its neighbours. This is the same trick AlphaFold's structure module
    uses, and it's the direct inverse of what build_frames() does.
    """

    def __init__(self, code_dim: int = 128, dim: int = 128, num_heads: int = 4, num_layers: int = 4):
        super().__init__()
        self.dim = dim

        self.input_proj = nn.Linear(code_dim, dim)
        self.layers = nn.ModuleList([TransformerLayer(dim, num_heads) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(dim)

        # Three output heads, one per thing a residue's geometry needs.
        self.to_rotation = nn.Linear(dim, 6)     # -> rotation, via rotation_from_6d
        self.to_translation = nn.Linear(dim, 3)  # -> where this residue's CA sits
        self.to_psi = nn.Linear(dim, 2)          # -> unnormalized (cos, sin) for O's swing

        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

        # Start every head predicting something neutral and sane: zero
        # weights, and biases set so that before any training every residue
        # comes out as an unrotated frame at the origin. Untrained random
        # rotations would make the first few gradient steps chaotic.
        for head, bias in [
            (self.to_rotation, [1.0, 0.0, 0.0, 0.0, 1.0, 0.0]),  # -> identity rotation
            (self.to_translation, [0.0, 0.0, 0.0]),
            (self.to_psi, [1.0, 0.0]),                            # -> angle 0
        ]:
            nn.init.zeros_(head.weight)
            with torch.no_grad():
                head.bias.copy_(torch.tensor(bias))

        # Ideal geometry is constant, so register it as buffers: moved to
        # GPU with .to(device) alongside the weights, but never trained.
        for name, tensor in ideal_local_backbone().items():
            self.register_buffer(f"ideal_{name}", tensor)

    def place_oxygen(self, psi: torch.Tensor) -> torch.Tensor:
        """
        Work out O's position in the local frame, given a predicted angle.

        psi: (B, L, 2) -- unnormalized (cos, sin)
        Returns: (B, L, 3) -- O in local-frame coordinates

        O is the one backbone atom the frame does not pin down. N, CA and C
        define the frame, so they're fixed inside it, but O hangs off the C
        atom and is free to swing around the CA->C axis. That swing is the
        psi torsion, and it genuinely varies between residues, so it has to
        be predicted rather than assumed.

        The network outputs a raw (cos, sin) pair which gets normalized to
        unit length, instead of outputting the angle directly. An angle
        would wrap around at 2*pi, and the network would have to make a
        discontinuous jump to cross that seam. A point on a circle has no
        seam. (Whatever constant offset there is between this angle and the
        formal definition of psi just gets absorbed into the learned
        weights, so it doesn't need to be worked out here.)
        """
        psi = psi / psi.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        cos_t, sin_t = psi[..., 0:1], psi[..., 1:2]

        # Direction from C to O: a fixed tilt off the CA->C axis (set by the
        # CA-C-O bond angle), plus a free rotation around it.
        swing = cos_t * self.ideal_p + sin_t * self.ideal_q
        direction = -math.cos(ANGLE_CA_C_O) * self.ideal_u + math.sin(ANGLE_CA_C_O) * swing

        return self.ideal_c + BOND_C_O * direction

    def forward(self, quantized: torch.Tensor, mask: torch.Tensor) -> dict:
        """
        quantized: (B, L, code_dim) -- the quantizer's straight-through output.
                   To decode from raw token ids instead, look them up first:
                   quantizer.codebook(tokens)
        mask:      (B, L) -- True for real residues

        Returns a dict with:
          coords:       (B, L, 4, 3) reconstructed N, CA, C, O positions
          rotations:    (B, L, 3, 3) predicted frames
          translations: (B, L, 3)    predicted CA positions
        """
        B, L, _ = quantized.shape

        x = self.input_proj(quantized)
        x = x + sinusoidal_positions(L, self.dim, x.device, x.dtype)

        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_norm(x)

        rotations = rotation_from_6d(self.to_rotation(x))  # (B, L, 3, 3)
        translations = self.to_translation(x)              # (B, L, 3)

        # Assemble each residue's four atoms in its own local frame: three
        # of them fixed by ideal chemistry, O from the predicted torsion.
        oxygen = self.place_oxygen(self.to_psi(x))                     # (B, L, 3)
        n = self.ideal_n.expand(B, L, 3)
        ca = self.ideal_ca.expand(B, L, 3)
        c = self.ideal_c.expand(B, L, 3)
        local_atoms = torch.stack([n, ca, c, oxygen], dim=2)           # (B, L, 4, 3)

        # Push each residue's local atoms out into shared world space:
        # rotate by that residue's frame, then shift to its CA position.
        # Exactly the inverse of to_local_frame().
        coords = torch.einsum("blij,blaj->blai", rotations, local_atoms) + translations.unsqueeze(2)

        return {"coords": coords, "rotations": rotations, "translations": translations}


def fape_loss(
    pred: dict,
    true_coords: torch.Tensor,
    mask: torch.Tensor,
    clamp: float = 10.0,
    length_scale: float = 10.0,
) -> torch.Tensor:
    """
    Frame Aligned Point Error -- the reconstruction loss, and the only kind
    that can work here.

    A plain MSE between predicted and true coordinates would be unlearnable.
    The encoder's features are rotation-invariant by construction, so the
    tokens carry no information at all about which way the protein was
    pointing in the original PDB file. A coordinate MSE would keep punishing
    the decoder for a global orientation it has no way to know, and the best
    it could do is hedge toward a blurry average.

    FAPE removes that problem by never comparing world positions. Instead,
    for every residue i, it re-expresses every atom in residue i's own local
    frame -- once using the predicted structure's frame and atoms, once
    using the true structure's -- and compares those. Rotate the whole true
    protein and every one of those local views is unchanged, so the loss is
    unchanged. What it's really measuring is "from where residue i is
    standing, does the rest of the protein look right," summed over every
    residue's point of view. Getting that right for all i leaves only one
    possible shape, so it's a strict measure despite ignoring orientation.

    clamp: errors beyond 10A stop growing. Early in training everything is
    wildly wrong, and without the clamp a few far-apart residue pairs would
    produce enormous gradients that drown out the many nearly-correct local
    ones. Capping the penalty keeps the model working on local geometry
    first and global arrangement second.

    length_scale: divides the result so the loss lands in [0, 1] rather than
    in Angstroms, which keeps it on a comparable footing with the
    quantizer's codebook and commitment losses when they're added together.
    """
    B, L = mask.shape

    true_rotations, true_translations = build_frames(true_coords)

    # Flatten the per-residue atom axis: every one of the L*4 atoms will be
    # viewed from every one of the L frames.
    pred_atoms = pred["coords"].reshape(B, L * 4, 3)
    true_atoms = true_coords.reshape(B, L * 4, 3)

    # Broadcast the atom list across the frame axis, then reuse the encoder's
    # own to_local_frame -- the same machinery on both sides guarantees the
    # two views are defined identically.
    pred_local = to_local_frame(
        pred_atoms.unsqueeze(1).expand(-1, L, -1, -1), pred["rotations"], pred["translations"]
    )  # (B, L, L*4, 3)
    true_local = to_local_frame(
        true_atoms.unsqueeze(1).expand(-1, L, -1, -1), true_rotations, true_translations
    )

    # Distance between the two views of each (frame, atom) pair. The epsilon
    # is load-bearing: .norm() of an exactly-zero vector has an undefined
    # (NaN) gradient, and predicted-equals-true happens for real.
    diff = pred_local - true_local
    distance = (diff.pow(2).sum(dim=-1) + 1e-8).sqrt()  # (B, L, L*4)
    distance = distance.clamp(max=clamp)

    # A pair counts only if both the frame residue and the atom's residue
    # are real. repeat_interleave(4) stretches the per-residue mask over
    # that residue's four atoms.
    frame_mask = mask.unsqueeze(2)                             # (B, L, 1)
    atom_mask = mask.repeat_interleave(4, dim=1).unsqueeze(1)  # (B, 1, L*4)
    pair_mask = (frame_mask & atom_mask).float()               # (B, L, L*4)

    return (distance * pair_mask).sum() / pair_mask.sum().clamp(min=1.0) / length_scale


In [ ]:
%%writefile data/fetch_pdb_ids.py
"""
Query RCSB's search API for single-chain protein structures with
50-300 residues, and save the resulting PDB IDs to a text file.

Output of this script is the --id-list input for download_pdbs.py.
"""

import argparse
from pathlib import Path

import requests

# RCSB's structured search API (separate from the file-download endpoint
# used in download_pdbs.py). Takes a JSON query, returns matching entry IDs.
SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"


def build_query(min_length: int, max_length: int, max_results: int) -> dict:
    """Build the RCSB search request body for our filter criteria."""
    return {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    # Only entries that are protein-only (no nucleic acids, no ligand-only entries)
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.selected_polymer_entity_types",
                        "operator": "exact_match",
                        "value": "Protein (only)",
                    },
                },
                {
                    # Exactly one distinct protein entity -> single chain, not a complex
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.polymer_entity_count_protein",
                        "operator": "equals",
                        "value": 1,
                    },
                },
                {
                    # Residue count lower bound. Note: polymer_monomer_count (no
                    # min/max suffix) exists in RCSB's schema but isn't search-enabled;
                    # the queryable per-entry fields are the _minimum/_maximum variants.
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.polymer_monomer_count_minimum",
                        "operator": "greater_or_equal",
                        "value": min_length,
                    },
                },
                {
                    # Residue count upper bound
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.polymer_monomer_count_maximum",
                        "operator": "less_or_equal",
                        "value": max_length,
                    },
                },
            ],
        },
        "return_type": "entry",  # we want PDB entry IDs, not entity/assembly IDs
        "request_options": {
            "paginate": {"start": 0, "rows": max_results},
            "results_content_type": ["experimental"],  # skip computed/predicted models
        },
    }


def fetch_pdb_ids(min_length: int, max_length: int, max_results: int) -> list[str]:
    """Run the search query and return a list of matching PDB IDs."""
    query = build_query(min_length, max_length, max_results)
    response = requests.post(SEARCH_URL, json=query, timeout=30)
    response.raise_for_status()

    data = response.json()
    return [hit["identifier"] for hit in data.get("result_set", [])]


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--min-length", type=int, default=50)
    parser.add_argument("--max-length", type=int, default=300)
    parser.add_argument("--max-results", type=int, default=1000)
    parser.add_argument(
        "--out-file",
        type=Path,
        default=Path("data/pdb_ids.txt"),
        help="Where to save the resulting list of PDB IDs.",
    )
    args = parser.parse_args()

    print(f"Querying RCSB for single-chain proteins, {args.min_length}-{args.max_length} residues...")
    pdb_ids = fetch_pdb_ids(args.min_length, args.max_length, args.max_results)
    print(f"Found {len(pdb_ids)} matching entries.")

    args.out_file.parent.mkdir(parents=True, exist_ok=True)
    args.out_file.write_text("\n".join(pdb_ids))
    print(f"Saved to {args.out_file}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile data/download_pdbs.py
"""
Download raw .pdb files from RCSB given a list of PDB IDs.
This is step 1 of the structure-tokenizer pipeline.
"""  # module docstring: becomes __doc__, shown by --help later

import argparse          # stdlib: parses command-line flags like --id-list
import time               # stdlib: gives us time.sleep() to throttle requests
from pathlib import Path  # stdlib: object-oriented filesystem paths (vs raw strings)

import requests  # 3rd-party (pip install requests): makes HTTP GET calls

RCSB_URL_TEMPLATE = "https://files.rcsb.org/download/{pdb_id}.pdb"  # URL pattern; {pdb_id} filled in later
REQUEST_DELAY_SECONDS = 0.2  # seconds to sleep between downloads, avoids hammering RCSB


def read_pdb_id_list(list_path: Path) -> list[str]:      # takes a Path, returns a list of strings
    """Load PDB IDs from a plain text file, one ID per line."""  # docstring for this function
    with open(list_path) as f:                            # open file; auto-closes when block ends
        ids = [                                            # start building a list via comprehension
            line.strip().upper()                           # per kept line: strip whitespace/newline, uppercase it
            for line in f                                  # iterate the file object one line at a time
            if line.strip()                                # skip blank lines (empty string is falsy)
        ]
    return ids                                              # hand back the cleaned list of IDs


def download_one(pdb_id: str, out_dir: Path, session: requests.Session) -> bool:  # one ID -> True/False
    """Download a single PDB file. Returns True on success, False on failure."""  # docstring
    out_path = out_dir / f"{pdb_id}.pdb"    # Path "/" joins segments -> e.g. data/raw_pdb/1ABC.pdb

    if out_path.exists():        # already downloaded in a previous run?
        return True               # treat as success, skip re-downloading (makes script resumable)

    url = RCSB_URL_TEMPLATE.format(pdb_id=pdb_id)  # substitute {pdb_id} into the URL template
    try:                                             # begin error-handling block
        response = session.get(url, timeout=15)      # HTTP GET; give up after 15s with no response
        response.raise_for_status()                  # raise an exception if status code is 4xx/5xx
    except requests.RequestException as e:            # catch timeout / connection error / bad status
        print(f"  FAILED {pdb_id}: {e}")               # log which ID failed and why
        return False                                    # signal failure to caller

    out_path.write_bytes(response.content)  # write raw response bytes to disk (bytes, not text — avoids encoding issues)
    return True                              # signal success to caller


def main():                                     # entry point, called only when script is run directly
    parser = argparse.ArgumentParser(description=__doc__)  # create CLI parser; reuse module docstring as help text

    parser.add_argument(              # register the --id-list flag
        "--id-list",                   # flag name as typed on the command line
        type=Path,                     # argparse wraps the given string in a Path automatically
        required=True,                 # script errors out if this flag is missing
        help="Text file with one PDB ID per line (e.g. 1ABC).",  # shown in --help
    )
    parser.add_argument(              # register the --out-dir flag
        "--out-dir",                   # flag name
        type=Path,                     # again auto-wrapped as a Path
        default=Path("data/raw_pdb"),  # used if the flag is omitted
        help="Directory to save downloaded .pdb files into.",  # shown in --help
    )
    args = parser.parse_args()  # actually read sys.argv and populate args.id_list / args.out_dir

    args.out_dir.mkdir(parents=True, exist_ok=True)  # create output dir (and any missing parents); no error if it exists
    pdb_ids = read_pdb_id_list(args.id_list)          # load the list of IDs from the given file
    print(f"Downloading {len(pdb_ids)} PDB files to {args.out_dir}/")  # status message to the user

    session = requests.Session()  # reuse one TCP connection across all ~1000 requests instead of opening a new one each time

    failed = []                                    # collect IDs that failed, to report/save at the end
    for i, pdb_id in enumerate(pdb_ids, start=1):   # loop with a 1-based counter i alongside each pdb_id
        ok = download_one(pdb_id, args.out_dir, session)  # attempt the download, get True/False back
        if not ok:                                   # if it failed...
            failed.append(pdb_id)                     # ...record it
        if i % 50 == 0:                              # every 50th item...
            print(f"  {i}/{len(pdb_ids)} processed...")  # ...print a progress update
        time.sleep(REQUEST_DELAY_SECONDS)             # pause briefly before the next request

    print(f"Done. {len(pdb_ids) - len(failed)} succeeded, {len(failed)} failed.")  # final summary
    if failed:                                          # if anything failed...
        failed_path = args.out_dir / "_failed_ids.txt"    # ...build a path for a failure-log file
        failed_path.write_text("\n".join(failed))          # ...write one failed ID per line
        print(f"Failed IDs written to {failed_path}")       # ...tell the user where to find it


if __name__ == "__main__":  # True only when this file is run directly (not imported as a module)
    main()                    # kick off the whole script


In [ ]:
%%writefile data/parse_structures.py
"""
Parse raw .pdb files into backbone coordinate arrays for structure
tokenizer training.

For each structure, extracts the (N, CA, C, O) backbone atom coordinates
per residue and the amino acid sequence, then saves them together as a
single .npz file. This is step 3 of the pipeline: raw .pdb files (from
download_pdbs.py) -> clean per-residue coordinate arrays (this script)
-> PyTorch Dataset (next step).
"""

import argparse
from pathlib import Path

import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1

# The four backbone atoms every standard amino acid residue has. Together
# they define the protein's fold; side-chain atoms vary per residue type
# and aren't needed for a backbone-level structure tokenizer.
BACKBONE_ATOMS = ["N", "CA", "C", "O"]


def extract_backbone(pdb_path: Path, min_length: int) -> tuple[np.ndarray, str] | None:
    """
    Parse one .pdb file and return (coords, sequence), or None if the
    structure doesn't yield a usable single chain.

    coords has shape (L, 4, 3): L residues x 4 backbone atoms x xyz.
    sequence is the one-letter amino acid code string, length L.
    """
    # QUIET=True suppresses BioPython's warnings about minor format
    # quirks (common in real PDB files) that we don't need to see per-file.
    parser = PDBParser(QUIET=True)

    try:
        structure = parser.get_structure(pdb_path.stem, pdb_path)
    except Exception:
        # Malformed file (truncated download, unparseable header, etc.)
        return None

    # Some PDB entries (e.g. NMR structures) contain multiple models of
    # the same molecule. We only want one static structure per entry, so
    # take the first model and ignore the rest.
    model = structure[0]

    # We filtered for single-chain entries when building the ID list, but
    # some entries still have a water/ligand "chain" alongside the real
    # one. Grab the first chain that actually contains amino acids.
    chain = None
    for candidate in model:
        if any(is_aa(res, standard=True) for res in candidate):
            chain = candidate
            break
    if chain is None:
        return None

    coords = []
    sequence = []
    for residue in chain:
        # Skip anything that isn't a standard amino acid: waters, metal
        # ions, ligands, and modified/non-standard residues all show up
        # as separate "residues" in BioPython's model but aren't part of
        # the sequence we want to tokenize.
        if not is_aa(residue, standard=True):
            continue

        # Crystal structures often have missing density for some atoms
        # (flexible loops, disordered regions). If any backbone atom is
        # absent for this residue, we can't get a coordinate for it, so
        # we drop the whole residue rather than leaving a gap or faking
        # a value.
        if not all(atom in residue for atom in BACKBONE_ATOMS):
            continue

        residue_coords = [residue[atom].get_coord() for atom in BACKBONE_ATOMS]
        coords.append(residue_coords)
        sequence.append(seq1(residue.get_resname()))

    # After dropping incomplete residues, the chain may have shrunk below
    # our length floor (e.g. a 55-residue entry losing 10 disordered
    # residues). Re-check here rather than trusting the original RCSB filter.
    if len(coords) < min_length:
        return None

    coords_array = np.array(coords, dtype=np.float32)  # shape (L, 4, 3)
    sequence_str = "".join(sequence)
    return coords_array, sequence_str


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--pdb-dir",
        type=Path,
        default=Path("data/raw_pdb"),
        help="Directory of raw .pdb files (output of download_pdbs.py).",
    )
    parser.add_argument(
        "--out-dir",
        type=Path,
        default=Path("data/parsed"),
        help="Directory to save parsed .npz files into.",
    )
    parser.add_argument(
        "--min-length",
        type=int,
        default=50,
        help="Drop structures shorter than this after removing incomplete residues.",
    )
    args = parser.parse_args()

    args.out_dir.mkdir(parents=True, exist_ok=True)
    pdb_files = sorted(args.pdb_dir.glob("*.pdb"))
    print(f"Parsing {len(pdb_files)} structures from {args.pdb_dir}/")

    failed = []
    succeeded = 0
    for i, pdb_path in enumerate(pdb_files, start=1):
        out_path = args.out_dir / f"{pdb_path.stem}.npz"

        # Same resumability pattern as download_pdbs.py: skip work already done.
        if out_path.exists():
            succeeded += 1
            continue

        result = extract_backbone(pdb_path, args.min_length)
        if result is None:
            failed.append(pdb_path.stem)
            continue

        coords, sequence = result
        # savez (not savez_compressed) — these arrays are small (a few KB
        # each), so compression overhead isn't worth the slower read speed
        # during training, when this file gets loaded repeatedly.
        np.savez(out_path, coords=coords, sequence=sequence)
        succeeded += 1

        if i % 100 == 0:
            print(f"  {i}/{len(pdb_files)} processed...")

    print(f"Done. {succeeded} succeeded, {len(failed)} failed/skipped.")
    if failed:
        failed_path = args.out_dir / "_failed_ids.txt"
        failed_path.write_text("\n".join(failed))
        print(f"Failed IDs written to {failed_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile data/dataset.py
"""
PyTorch Dataset over parsed protein structures (.npz files produced by
parse_structures.py). Step 4 of the pipeline: parsed coordinate arrays
-> batches ready to feed into a model.
"""

from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

# The 20 standard amino acids, in a fixed order. Every sequence letter
# gets mapped to its index in this string (e.g. "A" -> 0, "C" -> 1, ...).
# The model works with these integer indices, not raw letters.
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# Reserved index for padding -- positions we add just to make a batch
# rectangular, not real residues. Placed after the 20 real amino acids.
PAD_INDEX = len(AMINO_ACIDS)


class StructureDataset(Dataset):
    """
    One item = one parsed protein structure: backbone coordinates plus
    its amino acid sequence, both loaded from a single .npz file.
    """

    def __init__(self, parsed_dir: Path):
        self.parsed_dir = Path(parsed_dir)
        # Build the file list once, at startup, rather than re-scanning
        # the directory on every __getitem__ call.
        self.files = sorted(self.parsed_dir.glob("*.npz"))

    def __len__(self):
        # PyTorch's DataLoader calls this to know how many items exist,
        # e.g. to decide how many batches make up one epoch.
        return len(self.files)

    def __getitem__(self, idx):
        # PyTorch calls this with an integer index whenever it needs one
        # specific example -- this is where the actual file gets read.
        data = np.load(self.files[idx])
        coords = torch.from_numpy(data["coords"])  # (L, 4, 3) float32

        # sequence was saved as a numpy string; convert each letter to
        # its integer index using the lookup table above.
        sequence_str = str(data["sequence"])
        sequence = torch.tensor(
            [AA_TO_INDEX[aa] for aa in sequence_str], dtype=torch.long
        )

        return {"coords": coords, "sequence": sequence}


def collate_fn(batch: list[dict]) -> dict:
    """
    Combine a list of variable-length examples into one padded batch.

    Structures have different numbers of residues (L varies per protein),
    but a batch tensor needs one fixed shape. We pad every example up to
    the longest one in this batch, and return a mask marking which
    positions are real residues vs. padding.
    """
    lengths = [item["coords"].shape[0] for item in batch]
    max_len = max(lengths)
    batch_size = len(batch)

    # Pre-allocate zero-filled tensors of the final padded shape, then
    # fill in each example's real data. Padded positions stay zero.
    coords_batch = torch.zeros(batch_size, max_len, 4, 3)
    sequence_batch = torch.full((batch_size, max_len), PAD_INDEX, dtype=torch.long)
    # mask is True for real residues, False for padding -- models use this
    # to ignore padded positions in attention/loss computations.
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)

    for i, item in enumerate(batch):
        length = lengths[i]
        coords_batch[i, :length] = item["coords"]
        sequence_batch[i, :length] = item["sequence"]
        mask[i, :length] = True

    return {"coords": coords_batch, "sequence": sequence_batch, "mask": mask}


In [ ]:
%%writefile train.py
"""
Training loop: the file that actually turns the encoder, quantizer and
decoder into a working structure tokenizer.

The whole thing is one loop over protein structures, doing:

    backbone coords -> encoder -> continuous vectors
                    -> quantizer -> discrete tokens (+ VQ loss)
                    -> decoder -> rebuilt backbone coords
                    -> FAPE loss vs. the original

and then nudging all three parts to make that round trip better. Nothing
supervises the tokens directly; they only have to be good enough that the
decoder can rebuild the shape from them. That reconstruction pressure is
the entire training signal.

Usage:
    python train.py --parsed-dir data/parsed --epochs 100

Resume an interrupted run:
    python train.py --resume checkpoints/last.pt
"""

import argparse
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Sampler, Subset

from data.dataset import StructureDataset, collate_fn
from model.decoder import StructureDecoder, fape_loss
from model.encoder import StructureEncoder
from model.quantizer import VectorQuantizer


def set_seed(seed: int) -> None:
    """Make a run repeatable -- same data order, same weight init."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(requested: str) -> torch.device:
    """cuda if there's a GPU, else Apple's mps, else cpu."""
    if requested != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def structure_lengths(files: list[Path]) -> list[int]:
    """
    Read just the residue count of every structure, once, at startup.

    Needed up front so batches can be assembled by length (see below).
    Cheap enough to brute-force for a few thousand files, and the result
    is cached to disk so restarts don't repeat it.
    """
    lengths = []
    for path in files:
        with np.load(path) as data:
            lengths.append(int(data["coords"].shape[0]))
    return lengths


class QuadraticBudgetSampler(Sampler):
    """
    Decides which structures go together in a batch.

    A fixed batch size does not work for this model. The encoder builds a
    feature for every PAIR of residues, so its memory grows with L squared,
    not with L. A batch of 8 proteins of 100 residues is comfortable; a
    batch of 8 proteins of 400 residues is 16x the memory and will run out.
    Picking a batch size small enough for the worst case would waste most
    of the GPU on the many short structures.

    So instead of a fixed count, batches are filled up to a fixed budget of
    (batch size) x (longest structure in the batch) squared. Short proteins
    come many at a time, long ones a few at a time, and peak memory stays
    roughly constant either way.

    Structures are also sorted by length before batching, so each batch
    holds proteins of similar size and very little of it is wasted padding.
    A little random noise is added to the sort key, and the finished
    batches are shuffled, so the model doesn't see the exact same groupings
    in the same order every epoch.
    """

    def __init__(self, lengths: list[int], budget: int, max_length: int, shuffle: bool = True, seed: int = 0):
        # Structures longer than max_length get cropped before they reach
        # the model, so their memory cost is capped at the crop size.
        self.effective = [min(length, max_length) for length in lengths]
        self.budget = budget
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self._batches = self._build(seed)

    def _build(self, seed: int) -> list[list[int]]:
        rng = np.random.default_rng(seed)
        indices = list(range(len(self.effective)))

        if self.shuffle:
            # Jitter the sort key by up to +/-10% so batch groupings vary
            # between epochs while staying length-homogeneous.
            noise = rng.uniform(0.9, 1.1, size=len(indices))
            indices.sort(key=lambda i: self.effective[i] * noise[i])
        else:
            indices.sort(key=lambda i: self.effective[i])

        batches, current, longest = [], [], 0
        for i in indices:
            candidate_longest = max(longest, self.effective[i])
            # Would adding this structure push the batch over budget?
            if current and (len(current) + 1) * candidate_longest ** 2 > self.budget:
                batches.append(current)
                current, longest = [i], self.effective[i]
            else:
                current.append(i)
                longest = candidate_longest
        if current:
            batches.append(current)

        if self.shuffle:
            rng.shuffle(batches)
        return batches

    def set_epoch(self, epoch: int) -> None:
        """Re-roll the groupings each epoch, reproducibly."""
        self.epoch = epoch
        self._batches = self._build(self.seed + epoch)

    def __iter__(self):
        return iter(self._batches)

    def __len__(self):
        return len(self._batches)


def make_collate(max_length: int, train: bool):
    """
    Wrap the dataset's collate_fn with cropping.

    Very long chains are cut down to max_length before batching. This is
    standard for structure models and costs less than it sounds: local fold
    geometry is what the tokens are describing, and a 256-residue window
    contains plenty of it. During training the crop starts at a random
    offset, so over many epochs the model still sees every part of every
    long protein. During validation it's a fixed centre crop, so the
    validation number means the same thing every time it's computed.
    """

    def _collate(batch: list[dict]) -> dict:
        cropped = []
        for item in batch:
            length = item["coords"].shape[0]
            if length > max_length:
                if train:
                    start = random.randint(0, length - max_length)
                else:
                    start = (length - max_length) // 2
                item = {
                    "coords": item["coords"][start : start + max_length],
                    "sequence": item["sequence"][start : start + max_length],
                }
            cropped.append(item)
        return collate_fn(cropped)

    return _collate


@torch.no_grad()
def initialize_codebook(models, loader, device, num_codes) -> int:
    """
    Seed the codebook from real encoder outputs before training starts.

    The quantizer initializes its codebook to small random noise (norm about
    0.2), which is the textbook default, but the encoder's untrained outputs
    come out roughly a thousand times larger (norm about 227 -- it has no
    output normalization, and four residual layers keep adding to the signal).

    So on step 0 every single residue in the dataset is nearer to the same
    handful of codes than to any other, the VQ loss starts around 500 while
    the reconstruction loss is under 1, and the optimizer spends its first
    long stretch doing nothing but dragging 4096 codes across empty space.
    Reconstruction, the thing we actually care about, barely moves meanwhile.

    Dropping the codes onto actual encoder outputs instead starts them where
    the data already is. Same idea as the dead-code revival below, just done
    once for the whole codebook before the first step.
    """
    encoder, quantizer, _ = models
    encoder.eval()

    # A few batches is plenty to sample from; no need to sweep the dataset.
    pool = []
    collected = 0
    for batch in loader:
        latents = encoder(batch["coords"].to(device), batch["mask"].to(device))
        pool.append(latents[batch["mask"].to(device)])
        collected += pool[-1].shape[0]
        if collected >= num_codes * 4:
            break

    pool = torch.cat(pool)
    picks = torch.randint(0, pool.shape[0], (num_codes,), device=device)
    chosen = pool[picks]
    quantizer.codebook.weight.copy_(chosen + torch.randn_like(chosen) * 0.01)
    return collected


@torch.no_grad()
def revive_dead_codes(quantizer, encoder_output, mask, usage_counts) -> int:
    """
    Reset codebook entries that nothing is using.

    This is the characteristic failure mode of VQ-VAEs, and with 4096 codes
    and only ~1000 training structures it is close to guaranteed. A code
    that starts out far from every encoder output never gets chosen, so it
    never receives a gradient, so it never moves closer -- it's dead, and
    it stays dead. Left alone, a run can end up genuinely using a few
    hundred of its 4096 codes, and the tokenizer is far coarser than the
    codebook size suggests.

    The standard fix is blunt and works: every so often, find the codes
    that went unused and move them on top of actual encoder outputs from
    the current batch, where they'll be near real data and stand a chance
    of being picked. A little noise is added so that two codes revived from
    the same batch don't land in exactly the same place.
    """
    dead = (usage_counts == 0).nonzero().squeeze(-1)
    if dead.numel() == 0:
        return 0

    pool = encoder_output[mask]  # (num_real_residues, code_dim)
    if pool.shape[0] == 0:
        return 0

    picks = torch.randint(0, pool.shape[0], (dead.numel(),), device=pool.device)
    replacement = pool[picks]
    quantizer.codebook.weight[dead] = replacement + torch.randn_like(replacement) * 0.01
    return int(dead.numel())


def codebook_report(counts: torch.Tensor) -> dict:
    """
    Two numbers describing how much of the codebook is really in use.

    'used' is the blunt count of codes that appeared at least once.
    'perplexity' is the subtler one -- roughly "how many codes are in
    meaningful rotation." If usage is spread evenly over 500 codes,
    perplexity is about 500. If 490 of those are used once each and 10
    codes absorb everything else, 'used' still says 500 but perplexity
    drops to near 10, which is the honest answer.
    """
    total = counts.sum()
    if total == 0:
        return {"used": 0, "perplexity": 0.0}
    probs = counts.float() / total
    nonzero = probs[probs > 0]
    entropy = -(nonzero * nonzero.log()).sum()
    return {"used": int((counts > 0).sum()), "perplexity": float(entropy.exp())}


def run_epoch(models, loader, device, optimizer=None, scheduler=None, args=None,
              usage_since_revive=None, global_step=0):
    """
    One pass over the data. Training when an optimizer is given, evaluating
    when it isn't -- the forward pass is identical either way, which is the
    point of keeping them in one function.
    """
    encoder, quantizer, decoder = models
    training = optimizer is not None
    for module in models:
        module.train(training)

    totals = {"recon": 0.0, "vq": 0.0, "residues": 0}
    epoch_counts = torch.zeros(quantizer.codebook.num_embeddings, dtype=torch.long, device=device)
    revived = 0

    for batch in loader:
        global_step += 1
        coords = batch["coords"].to(device)
        mask = batch["mask"].to(device)

        with torch.set_grad_enabled(training):
            latents = encoder(coords, mask)
            quantized = quantizer(latents, mask)
            rebuilt = decoder(quantized["quantized"], mask)

            reconstruction_loss = fape_loss(rebuilt, coords, mask)
            loss = reconstruction_loss + quantized["loss"]

        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            # Clip gradients before stepping. Early in training a badly
            # wrong structure can produce a huge gradient that knocks the
            # weights somewhere they never recover from; clipping caps the
            # step size without changing its direction.
            torch.nn.utils.clip_grad_norm_(
                [p for m in models for p in m.parameters()], args.grad_clip
            )
            optimizer.step()
            scheduler.step()

        # Tally which codes got used, for the dead-code report and revival.
        used = quantized["tokens"][mask].detach()
        batch_counts = torch.bincount(used, minlength=epoch_counts.numel())
        epoch_counts += batch_counts
        if usage_since_revive is not None:
            usage_since_revive += batch_counts

        if training and args.revive_every > 0 and global_step % args.revive_every == 0:
            revived += revive_dead_codes(quantizer, latents.detach(), mask, usage_since_revive)
            usage_since_revive.zero_()

        # Weight each batch's loss by how many real residues it held, so
        # the epoch average isn't skewed by batches of different sizes.
        residues = int(mask.sum())
        totals["recon"] += reconstruction_loss.detach().item() * residues
        totals["vq"] += quantized["loss"].detach().item() * residues
        totals["residues"] += residues

    n = max(totals["residues"], 1)
    report = codebook_report(epoch_counts)
    return {
        "recon": totals["recon"] / n,
        # The same reconstruction error in Angstroms rather than in
        # normalized loss units, which is the number worth watching. It is
        # a clamped average, so it can never exceed 10 no matter how bad
        # the prediction is.
        "recon_angstrom": (totals["recon"] / n) * 10.0,
        "vq": totals["vq"] / n,
        "codes_used": report["used"],
        "perplexity": report["perplexity"],
        "revived": revived,
        "global_step": global_step,
    }


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--parsed-dir", type=Path, default=Path("data/parsed"))
    parser.add_argument("--checkpoint-dir", type=Path, default=Path("checkpoints"))
    parser.add_argument("--resume", type=Path, default=None)
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--weight-decay", type=float, default=0.01)
    parser.add_argument("--warmup-steps", type=int, default=500)
    parser.add_argument("--grad-clip", type=float, default=1.0)
    parser.add_argument("--val-fraction", type=float, default=0.1)
    parser.add_argument("--max-length", type=int, default=256, help="Crop structures longer than this.")
    parser.add_argument(
        "--budget",
        type=int,
        default=4 * 256 ** 2,
        help="Batch size limit as (count x longest^2). Halve it if you hit out-of-memory.",
    )
    parser.add_argument("--num-codes", type=int, default=4096)
    parser.add_argument("--dim", type=int, default=128)
    parser.add_argument("--num-heads", type=int, default=4)
    parser.add_argument("--num-layers", type=int, default=4)
    parser.add_argument(
        "--no-data-init",
        action="store_true",
        help="Skip seeding the codebook from real encoder outputs (not recommended).",
    )
    parser.add_argument("--revive-every", type=int, default=500, help="Steps between dead-code resets; 0 disables.")
    parser.add_argument("--num-workers", type=int, default=2, help="Use 0 if dataloader workers misbehave on macOS.")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--device", type=str, default="auto")
    args = parser.parse_args()

    set_seed(args.seed)
    device = pick_device(args.device)
    args.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    dataset = StructureDataset(args.parsed_dir)
    if len(dataset) == 0:
        raise SystemExit(
            f"No .npz files in {args.parsed_dir}. Run data/parse_structures.py first."
        )

    # Cache the length list -- reading every file's header takes a moment
    # and the answer never changes.
    cache_path = args.parsed_dir / "_lengths.json"
    names = [p.name for p in dataset.files]
    cached = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    if cached.get("names") != names:
        cached = {"names": names, "lengths": structure_lengths(dataset.files)}
        cache_path.write_text(json.dumps(cached))
    lengths = cached["lengths"]

    # Split by a seeded shuffle so the same structures are held out on
    # every run, including after a resume.
    permutation = np.random.default_rng(args.seed).permutation(len(dataset))
    num_val = max(1, int(len(dataset) * args.val_fraction))
    val_indices, train_indices = permutation[:num_val], permutation[num_val:]

    train_sampler = QuadraticBudgetSampler(
        [lengths[i] for i in train_indices], args.budget, args.max_length, shuffle=True, seed=args.seed
    )
    val_sampler = QuadraticBudgetSampler(
        [lengths[i] for i in val_indices], args.budget, args.max_length, shuffle=False
    )

    train_loader = DataLoader(
        Subset(dataset, train_indices.tolist()),
        batch_sampler=train_sampler,
        collate_fn=make_collate(args.max_length, train=True),
        num_workers=args.num_workers,
    )
    val_loader = DataLoader(
        Subset(dataset, val_indices.tolist()),
        batch_sampler=val_sampler,
        collate_fn=make_collate(args.max_length, train=False),
        num_workers=args.num_workers,
    )

    encoder = StructureEncoder(args.dim, args.num_heads, args.num_layers).to(device)
    quantizer = VectorQuantizer(args.num_codes, args.dim).to(device)
    decoder = StructureDecoder(args.dim, args.dim, args.num_heads, args.num_layers).to(device)
    models = (encoder, quantizer, decoder)

    parameters = [p for m in models for p in m.parameters()]
    optimizer = torch.optim.AdamW(parameters, lr=args.lr, weight_decay=args.weight_decay)

    # Linear warmup, then a slow cosine decay. The warmup matters more than
    # usual here: at step 0 the codebook is random noise, so the tokens are
    # meaningless and the decoder's gradients are pure nonsense. Easing in
    # stops those first few steps from doing lasting damage.
    total_steps = max(1, args.epochs * len(train_sampler))

    def lr_at(step: int) -> float:
        if step < args.warmup_steps:
            return step / max(1, args.warmup_steps)
        progress = (step - args.warmup_steps) / max(1, total_steps - args.warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)

    start_epoch, best_val = 0, float("inf")
    if args.resume and args.resume.exists():
        state = torch.load(args.resume, map_location=device)
        encoder.load_state_dict(state["encoder"])
        quantizer.load_state_dict(state["quantizer"])
        decoder.load_state_dict(state["decoder"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        start_epoch, best_val = state["epoch"] + 1, state["best_val"]
        print(f"Resumed from {args.resume} at epoch {start_epoch}")

    if not args.no_data_init and not (args.resume and args.resume.exists()):
        sampled = initialize_codebook(models, train_loader, device, args.num_codes)
        print(f"seeded codebook from {sampled} encoder outputs")

    num_params = sum(p.numel() for p in parameters)
    print(f"device: {device}   parameters: {num_params/1e6:.2f}M")
    print(f"structures: {len(train_indices)} train / {len(val_indices)} val")
    print(f"batches per epoch: {len(train_sampler)}")

    usage_since_revive = torch.zeros(args.num_codes, dtype=torch.long, device=device)
    # Counted across the whole run, not per epoch -- otherwise --revive-every
    # would silently never trigger on datasets with few batches per epoch.
    global_step = start_epoch * len(train_sampler)

    for epoch in range(start_epoch, args.epochs):
        train_sampler.set_epoch(epoch)
        started = time.time()

        train_stats = run_epoch(
            models, train_loader, device, optimizer, scheduler, args,
            usage_since_revive, global_step,
        )
        global_step = train_stats["global_step"]
        val_stats = run_epoch(models, val_loader, device)

        print(
            f"epoch {epoch:3d}  "
            f"train {train_stats['recon_angstrom']:.3f}A  "
            f"val {val_stats['recon_angstrom']:.3f}A  "
            f"vq {train_stats['vq']:.4f}  "
            f"codes {train_stats['codes_used']}/{args.num_codes}  "
            f"ppl {train_stats['perplexity']:.0f}  "
            f"revived {train_stats['revived']}  "
            f"{time.time() - started:.0f}s"
        )

        state = {
            "encoder": encoder.state_dict(),
            "quantizer": quantizer.state_dict(),
            "decoder": decoder.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "best_val": best_val,
            # Paths stringified so the checkpoint stays loadable under
            # torch.load's weights_only=True default (PyTorch 2.6+), which
            # refuses to unpickle a PosixPath.
            "args": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},
        }
        # Always overwrite last.pt so an interrupted run can resume; keep
        # best.pt separately so a late overfitting slide can't erase the
        # best model we actually found.
        torch.save(state, args.checkpoint_dir / "last.pt")
        if val_stats["recon_angstrom"] < best_val:
            best_val = state["best_val"] = val_stats["recon_angstrom"]
            torch.save(state, args.checkpoint_dir / "best.pt")

    print(f"Done. Best validation reconstruction: {best_val:.3f}A")


if __name__ == "__main__":
    main()


## 4. Verify the padding fix landed

`collate_fn` pads with zero coordinates. Without the clamps in
`build_frames`, that produces NaN which spreads into the *real* residues too,
and every batch containing padding goes to NaN loss. This cell proves the
loaded code is the fixed version before you spend GPU hours on it.

In [ ]:
import torch, importlib
import model.geometry, model.encoder, model.quantizer, model.decoder
for m in [model.geometry, model.encoder, model.quantizer, model.decoder]:
    importlib.reload(m)
from model.encoder import StructureEncoder
from model.quantizer import VectorQuantizer
from model.decoder import StructureDecoder, fape_loss

torch.manual_seed(0)
B, L = 2, 12
coords = torch.randn(B, L, 4, 3) * 5
mask = torch.ones(B, L, dtype=torch.bool)
coords[1, 7:] = 0.0        # exactly what collate_fn produces for padding
mask[1, 7:] = False

enc, vq, dec = StructureEncoder(), VectorQuantizer(), StructureDecoder()
z = enc(coords, mask); q = vq(z, mask); out = dec(q["quantized"], mask)
loss = fape_loss(out, coords, mask) + q["loss"]

assert torch.isfinite(loss), "NaN loss -- geometry.py is missing the clamp fix"
assert not torch.isnan(z[mask]).any(), "NaN leaked into real residues"
print(f"padded-batch loss = {loss.item():.4f}  (finite, good)")

## 5. Build the training set

Three steps: ask RCSB which structures to use, download them, parse them into
`.npz` coordinate arrays. All three skip work that's already done, so
re-running this cell after a restart is cheap.

This takes a while the first time. See section 6 for how to avoid ever
repeating it.

In [ ]:
import subprocess, sys
from pathlib import Path

def run(*args):
    """Run a pipeline step and stop loudly if it fails, instead of letting
    a silent failure look like an empty dataset three cells later."""
    print(">", " ".join(args), flush=True)
    result = subprocess.run([sys.executable, *args])
    if result.returncode != 0:
        raise SystemExit(f"step failed: {' '.join(args)}")

# Bump --max-results once the pipeline is proven. 1000 structures is enough
# to get a real signal and small enough to download in reasonable time.
if not Path("data/pdb_ids.txt").exists():
    run("data/fetch_pdb_ids.py", "--max-results", "1000",
        "--min-length", "50", "--max-length", "300")
else:
    print("data/pdb_ids.txt already exists, skipping the RCSB query")

run("data/download_pdbs.py", "--id-list", "data/pdb_ids.txt", "--out-dir", "data/raw_pdb")
run("data/parse_structures.py", "--pdb-dir", "data/raw_pdb",
    "--out-dir", "data/parsed", "--min-length", "50")

n = len(list(Path("data/parsed").glob("*.npz")))
print(f"\nparsed structures ready: {n}")
assert n > 0, "nothing parsed -- check the download output above"

## 6. Save the parsed data so you never rebuild it

Downloading a thousand PDB files every session is a waste of your GPU quota.
Run this once, then use *File -> Save Version* so `/kaggle/working` is kept.
After that, create a Kaggle Dataset from the output and attach it to the
notebook. On later runs, skip section 5 and run this instead.

In [ ]:
from pathlib import Path
import shutil

# If you've attached the parsed data as a Kaggle Dataset, point this at it.
# Adjust the folder name to whatever you called the dataset.
ATTACHED = Path("/kaggle/input/ira-esm-parsed")

if ATTACHED.exists():
    target = Path("data/parsed"); target.mkdir(parents=True, exist_ok=True)
    for f in ATTACHED.glob("*.npz"):
        if not (target / f.name).exists():
            shutil.copy(f, target / f.name)
    print("copied", len(list(target.glob('*.npz'))), "structures from the attached dataset")
else:
    print("no attached dataset found at", ATTACHED)
    print("using whatever section 5 built:",
          len(list(Path('data/parsed').glob('*.npz'))), "structures")

## 7. Train

Run in the background with `nohup` so a dropped browser connection doesn't
kill the run, and tail the log to watch it.

`--budget` is the memory control: batches are filled up to
(count x longest-protein-squared), because the encoder's cost grows with the
square of the length. 262144 is about 4 proteins of 256 residues at a time.
A single T4 handles that comfortably. Halve it if you hit out-of-memory,
double it if `nvidia-smi` shows the card idling.

Only one GPU is used. The model is small and the second T4 would cost more in
synchronisation than it returns.

In [ ]:
import subprocess, os

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

cmd = 'nohup python -u train.py \\\n  --parsed-dir data/parsed \\\n  --checkpoint-dir /kaggle/working/checkpoints \\\n  --epochs 200 \\\n  --max-length 256 \\\n  --budget 262144 \\\n  --lr 3e-4 \\\n  --num-workers 2 \\\n  --revive-every 200 \\\n  > /kaggle/working/train.log 2>&1 &'

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
subprocess.Popen(cmd, shell=True, env=env, cwd="/kaggle/working/ira-esm-tokenizer")
print("training started -- run the next cell to watch it")

In [ ]:
# Watch progress. Re-run this cell whenever you want an update.
!tail -n 25 /kaggle/working/train.log

### Reading the output

```
epoch  12  train 6.412A  val 6.388A  vq 0.0431  codes 3480/4096  ppl 2611  revived 4  38s
```

- **train / val** -- reconstruction error in Angstroms. The number to watch.
  It is a clamped average so it can never exceed 10, and it starts near 10.
- **vq** -- how far encoder outputs sit from their assigned codes. Should
  fall fast and stay small. If it stays large, the codebook is chasing the
  encoder rather than tracking it.
- **codes / ppl** -- how much of the codebook is genuinely in use.
  `codes` counts anything used at least once; `ppl` (perplexity) is the
  honest version. If 3000 codes fire once each and 20 absorb everything,
  `codes` says 3000 and `ppl` says about 20. Watch `ppl`.
- **revived** -- dead codes reset onto real data this epoch. High early,
  should settle toward zero.

## 8. Resuming after a session ends

Kaggle stops sessions on a time limit, and the GPU quota is weekly. The run
checkpoints every epoch, so pick up where it stopped:

1. *File -> Save Version* before the session ends, so `/kaggle/working` persists.
2. On the new session, run sections 1-4 and 6, then this cell.

In [ ]:
import subprocess, os
from pathlib import Path

ckpt = Path("/kaggle/working/checkpoints/last.pt")
assert ckpt.exists(), "no checkpoint found -- start a fresh run from section 7"

import torch
state = torch.load(ckpt, map_location="cpu")
print(f"resuming from epoch {state['epoch']}, best val so far {state['best_val']:.3f}A")

cmd = f'nohup python -u train.py \\\n  --parsed-dir data/parsed \\\n  --checkpoint-dir /kaggle/working/checkpoints \\\n  --resume /kaggle/working/checkpoints/last.pt \\\n  --epochs 200 --max-length 256 --budget 262144 --lr 3e-4 \\\n  --num-workers 2 --revive-every 200 \\\n  >> /kaggle/working/train.log 2>&1 &'
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"),
                 cwd="/kaggle/working/ira-esm-tokenizer")
print("resumed")

## 9. Plot the curves

In [ ]:
import re
import matplotlib.pyplot as plt

rows = []
for line in open("/kaggle/working/train.log"):
    m = re.match(r"epoch\s+(\d+)\s+train ([\d.]+)A\s+val ([\d.]+)A\s+vq ([\d.]+)\s+"
                 r"codes (\d+)/\d+\s+ppl (\d+)", line)
    if m:
        rows.append([float(x) for x in m.groups()])

assert rows, "no epoch lines in the log yet"
epoch, train, val, vq, codes, ppl = zip(*rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epoch, train, label="train"); axes[0].plot(epoch, val, label="val")
axes[0].set_title("reconstruction error"); axes[0].set_xlabel("epoch")
axes[0].set_ylabel("Angstroms"); axes[0].legend()

axes[1].plot(epoch, vq, color="tab:red"); axes[1].set_yscale("log")
axes[1].set_title("VQ loss (log scale)"); axes[1].set_xlabel("epoch")

axes[2].plot(epoch, codes, label="codes used")
axes[2].plot(epoch, ppl, label="perplexity")
axes[2].set_title("codebook usage"); axes[2].set_xlabel("epoch"); axes[2].legend()

plt.tight_layout(); plt.show()
print(f"best val: {min(val):.3f}A at epoch {int(epoch[val.index(min(val))])}")

## 10. Optional: normalize the encoder output

Not applied by default, because it changes `encoder.py`.

`StructureEncoder` has no output normalization, so its outputs come out with
norm around 227 while the codebook initializes around 0.23. The optimizer
spends its early effort dragging 4096 codes across empty space instead of
learning to reconstruct.

Measured over 120 epochs on synthetic data:

| | val error | VQ loss |
|---|---|---|
| as-is | 7.50 A | 4.4 |
| with final LayerNorm | **6.66 A** | **0.043** |

It also beat the unmodified version's 120-epoch result by epoch 35. Run this
cell before section 7 if you want it, then restart the run from scratch (do
not resume an old checkpoint into a changed architecture).

In [ ]:
from pathlib import Path

p = Path("model/encoder.py"); s = p.read_text()
if "self.final_norm" in s:
    print("already applied")
else:
    s = s.replace(
        "            [GeometricAttentionLayer(dim, num_heads) for _ in range(num_layers)]\n        )",
        "            [GeometricAttentionLayer(dim, num_heads) for _ in range(num_layers)]\n        )\n"
        "        # Keeps encoder outputs on the same scale as the codebook.\n"
        "        self.final_norm = nn.LayerNorm(dim)")
    s = s.replace(
        "            x = layer(x, pair_feats, mask)\n\n        return x",
        "            x = layer(x, pair_feats, mask)\n\n        return self.final_norm(x)")
    p.write_text(s)
    print("applied -- restart the kernel, re-run sections 1-6, then train fresh")